# Aplicación: diagnóstico de tumores de mama

## 2.1 Importar librerías
- `numpy`, `pandas`: manejo de datos numéricos y tablas.
- `matplotlib`, `seaborn`: gráficos.
- `load_breast_cancer`: conjunto de datos incluido en scikit-learn (569 pacientes, 30 medidas de las células).
- `train_test_split`: separa datos de entrenamiento y prueba.
- `StandardScaler`: estandariza las variables (media 0, desviación 1).
- `Pipeline`: encadena pasos (escalado y modelo) para evitar fugas de información.
- `SVC`: clasificador SVM.
- `GridSearchCV`: busca los mejores hiperparámetros con validación cruzada.
- `metrics`: herramientas de evaluación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)
from sklearn.inspection import DecisionBoundaryDisplay
import joblib

## 2.2 Cargar y explorar los datos
- `load_breast_cancer(as_frame=True)`: devuelve los datos como DataFrame de pandas.
- `X`: tabla con las 30 características (radio, textura, perímetro, área, etc.).
- `y`: etiqueta. En este dataset **0 = maligno** y **1 = benigno**.
- `value_counts()`: cuenta cuántos casos hay de cada clase para ver si están balanceadas.

In [ ]:
datos = load_breast_cancer(as_frame=True)
X = datos.data
y = datos.target
nombres_clases = ['Maligno', 'Benigno']   # 0 = maligno, 1 = benigno

print("Forma de X:", X.shape)
print("\nCasos por clase:")
print(y.map({0: 'Maligno', 1: 'Benigno'}).value_counts())
X.head()

## 2.3 Separar entrenamiento y prueba
- `train_test_split(..., test_size=0.2)`: 80% para entrenar y 20% para probar.
- `stratify=y`: mantiene la misma proporción de clases en ambos conjuntos.
- `random_state=42`: fija la aleatoriedad para obtener siempre el mismo resultado.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)

## 2.4 Primer modelo SVM (pipeline con escalado)
- `Pipeline([...])`: primero `StandardScaler` y después `SVC`. El escalador aprende la media y desviación **solo con los datos de entrenamiento**, evitando fuga de información hacia la prueba.
- `SVC(kernel='rbf', C=1, gamma='scale')`: SVM con kernel gaussiano.
- `fit()`: entrena el modelo.
- `predict()`: predice la clase de nuevos datos.
- `accuracy_score`: proporción de aciertos.
- `modelo['svc'].n_support_`: número de vectores de soporte por clase.

In [ ]:
modelo_base = Pipeline([
    ('escalador', StandardScaler()),
    ('svc', SVC(kernel='rbf', C=1, gamma='scale'))
])

modelo_base.fit(X_train, y_train)
pred = modelo_base.predict(X_test)

print(f"Precisión (modelo base): {accuracy_score(y_test, pred)*100:.2f}%")
print("Vectores de soporte por clase:", modelo_base['svc'].n_support_)

## 2.5 Comparar kernels
- Se recorre una lista de kernels y se entrena un modelo con cada uno.
- `cross_val_score(..., cv=5)`: validación cruzada de 5 pliegues; entrena y evalúa 5 veces con distintas particiones, dando una estimación más confiable que una sola división.
- Se muestra la media y la desviación estándar de la precisión.

In [ ]:
resultados = {}
for kernel in ['linear', 'poly', 'rbf', 'sigmoid']:
    pipe = Pipeline([('escalador', StandardScaler()),
                     ('svc', SVC(kernel=kernel))])
    puntajes = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    resultados[kernel] = puntajes
    print(f"{kernel:8s} -> {puntajes.mean()*100:.2f}% (+/- {puntajes.std()*100:.2f})")

plt.figure(figsize=(7, 4))
plt.boxplot(resultados.values(), labels=resultados.keys())
plt.title('Precisión por kernel (validación cruzada)')
plt.ylabel('Precisión')
plt.show()

## 2.6 Optimizar hiperparámetros con GridSearchCV
- `param_grid`: diccionario con los valores a probar. El prefijo `svc__` indica que el parámetro pertenece al paso `svc` del pipeline.
- `GridSearchCV`: prueba todas las combinaciones con validación cruzada y se queda con la mejor.
- `n_jobs=-1`: usa todos los núcleos del procesador.
- `best_params_`: mejores hiperparámetros; `best_score_`: mejor precisión en validación cruzada.

In [ ]:
pipe = Pipeline([('escalador', StandardScaler()), ('svc', SVC())])

param_grid = {
    'svc__kernel': ['linear', 'rbf'],
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.001, 0.01, 0.1]
}

busqueda = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
busqueda.fit(X_train, y_train)

print("Mejores parámetros:", busqueda.best_params_)
print(f"Mejor precisión en validación cruzada: {busqueda.best_score_*100:.2f}%")
mejor_modelo = busqueda.best_estimator_

## 2.7 Evaluar el mejor modelo con datos de prueba
- `classification_report`: precisión (precision), exhaustividad (recall) y F1 por clase.
- `confusion_matrix`: aciertos y errores por clase. En medicina el error más grave es el **falso negativo** (decir benigno a un tumor maligno), por eso se revisa el *recall* de la clase Maligno.
- `decision_function`: distancia con signo al hiperplano; se usa para calcular el **AUC-ROC**, que mide la capacidad de separar las clases (1.0 es perfecto).

In [ ]:
pred = mejor_modelo.predict(X_test)

print(f"Precisión en prueba: {accuracy_score(y_test, pred)*100:.2f}%")
print(f"AUC-ROC: {roc_auc_score(y_test, mejor_modelo.decision_function(X_test)):.4f}\n")
print(classification_report(y_test, pred, target_names=nombres_clases, digits=3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=nombres_clases).plot(cmap='Blues', colorbar=False)
plt.title('Matriz de confusión')
plt.show()

## 2.8 Visualizar la frontera de decisión y el efecto de C y gamma
- Para poder dibujar en 2D se usan solo dos características: `mean radius` y `mean texture`.
- Se entrena un SVM RBF con distintas combinaciones de C y gamma.
- `DecisionBoundaryDisplay.from_estimator`: colorea las regiones que el modelo asigna a cada clase.
- `plt.scatter`: dibuja los pacientes. Observa cómo con **gamma grande y C grande** la frontera se vuelve irregular (sobreajuste), y con valores pequeños es más suave.

In [ ]:
X2 = X[['mean radius', 'mean texture']].values
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, stratify=y, random_state=42)

combinaciones = [(0.1, 0.01), (1, 0.1), (10, 1), (100, 10)]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (C, gamma) in zip(axes, combinaciones):
    clf = Pipeline([('escalador', StandardScaler()),
                    ('svc', SVC(kernel='rbf', C=C, gamma=gamma))])
    clf.fit(X2_train, y2_train)
    DecisionBoundaryDisplay.from_estimator(clf, X2_train, response_method='predict',
                                           cmap='coolwarm', alpha=0.3, ax=ax,
                                           xlabel='Radio medio', ylabel='Textura media')
    ax.scatter(X2_train[:, 0], X2_train[:, 1], c=y2_train, cmap='coolwarm',
               edgecolors='k', s=15)
    acc = clf.score(X2_test, y2_test)
    ax.set_title(f"C={C}, gamma={gamma}\nPrueba: {acc*100:.1f}%")
plt.tight_layout()
plt.show()

## 2.9 Convertir el modelo en una aplicación
Para usarlo como aplicación:
1. `joblib.dump(modelo, archivo)`: guarda el modelo entrenado (incluido el escalador) en un archivo `.joblib`.
2. `joblib.load(archivo)`: lo carga en cualquier otro programa sin reentrenar.
3. La función `diagnosticar()` recibe las 30 medidas de un paciente y devuelve el diagnóstico, la distancia al hiperplano (mientras mayor su valor absoluto, mayor la confianza) y una alerta si el caso es dudoso (cerca de la frontera).
4. `decision_function`: valor positivo = Benigno, negativo = Maligno.

In [ ]:
# Guardar y volver a cargar el modelo
joblib.dump(mejor_modelo, 'modelo_svm_cancer.joblib')
modelo_cargado = joblib.load('modelo_svm_cancer.joblib')

def diagnosticar(modelo, medidas, umbral_duda=0.5):
    """Recibe un paciente (dict, Series o fila de DataFrame) con las 30 medidas
    y devuelve el diagnóstico sugerido del modelo SVM."""
    fila = pd.DataFrame([medidas])[list(X.columns)]    # ordena las columnas
    clase = modelo.predict(fila)[0]
    distancia = modelo.decision_function(fila)[0]      # distancia al hiperplano
    dudoso = abs(distancia) < umbral_duda
    print(f"Diagnóstico sugerido: {nombres_clases[clase].upper()}")
    print(f"Distancia al hiperplano: {distancia:.2f}")
    if dudoso:
        print("ALERTA: caso cercano a la frontera, se recomienda revisión médica adicional.")
    return clase

# Ejemplo: tres pacientes del conjunto de prueba
for i in range(3):
    paciente = X_test.iloc[i]
    print(f"--- Paciente {i+1} (real: {nombres_clases[y_test.iloc[i]]}) ---")
    diagnosticar(modelo_cargado, paciente)
    print()